# TFT v2 — Groundwater Forecasting (Google Colab)

**Improvements over v1 (see notes at bottom):**
1. Per-district target normalization (biggest accuracy win)
2. Cyclic time encoding (sin/cos of month + day_of_year)
3. Log1p rainfall (heavy-tailed -> tame distribution)
4. Rolling gwl statistics (30/60/90 day mean+std)
5. Deeper model: d_model=192, 3 LSTM layers, 8 heads
6. Longer encoder (120 days = full season of context)
7. Cosine LR with warmup; Huber-weighted q50 loss
8. Per-district metric reporting

**Setup:** Runtime -> Change runtime type -> **T4 GPU**, then upload `tft_model.py` + `master_dataset.csv`.

In [ ]:
# ── STEP 1: Upload files ─────────────────────────────────────────────────────
from google.colab import files
print('Upload tft_model.py and master_dataset.csv')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
!pip install scikit-learn -q
print('Done')

In [ ]:
import sys, warnings, json, time, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from tft_model import TemporalFusionTransformer, QuantileLoss

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU found. Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# ─── CONFIG ──────────────────────────────────────────────────────────────────
ENCODER_LEN   = 120        # was 90 — longer context captures full seasonal cycle
DECODER_LEN   = 30
EVAL_STEPS    = [6, 13, 29]
D_MODEL       = 192        # was 128 — more capacity
N_HEADS       = 8          # was 4
LSTM_LAYERS   = 3          # was 2
DROPOUT       = 0.15
QUANTILES     = [0.1, 0.5, 0.9]
CAT_EMBED_DIM = 16         # was 8 — richer district embedding
BATCH_SIZE    = 128
MAX_EPOCHS    = 120
LR            = 1e-3
WARMUP_EPOCHS = 5
PATIENCE      = 20
GRAD_CLIP     = 1.0
TRAIN_END     = '2025-01-01'
SEED          = 42
Q50_LOSS_BOOST= 1.0        # was 1.5 — removed median bias that squeezed q10/q90 too narrow

torch.manual_seed(SEED)
np.random.seed(SEED)

OUT_DIR   = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)
DATA_PATH = Path('master_dataset.csv')

print(f'Config: d_model={D_MODEL}, encoder={ENCODER_LEN}, lstm_layers={LSTM_LAYERS}, heads={N_HEADS}')

In [ ]:
# ── Load & impute ─────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df = df.sort_values(['district','date']).reset_index(drop=True)
print(f'Shape: {df.shape}  |  Districts: {df["district"].nunique()}  |  {df["date"].min().date()} -> {df["date"].max().date()}')

SOIL_COLS     = ['sand_0cm','clay_0cm','silt_0cm','sand_10cm','clay_10cm','silt_10cm','sand_30cm','clay_30cm','silt_30cm']
SENTINEL_COLS = ['ndvi_mean','ndwi_mean','ndmi_mean','vv_db','vh_db','vh_vv_ratio']
LAG_COLS      = ['gwl_lag_7','gwl_lag_14','gwl_lag_30','gwl_7d_change','gwl_30d_change']
CGWB_COLS     = ['annual_recharge_bcm','annual_extraction_bcm','extractable_resource_bcm','stage_of_extraction_pct']

df[SOIL_COLS]     = df[SOIL_COLS].fillna(df[SOIL_COLS].median())
df[SENTINEL_COLS] = df[SENTINEL_COLS].fillna(df[SENTINEL_COLS].median())
df[LAG_COLS]      = df.groupby('district')[LAG_COLS].transform(lambda s: s.fillna(s.median()))
df[CGWB_COLS]     = df[CGWB_COLS].fillna(df[CGWB_COLS].median())     # safety; merge gave 0 nulls
if 'aquifer_category_id' in df.columns:
    df['aquifer_category_id'] = df['aquifer_category_id'].fillna(0).astype(int)
df['season_id']   = df['season'].map({'winter':0,'summer':1,'monsoon':2,'post_monsoon':3}).astype(int)
print('Missing after imputation:', df.isnull().sum().sum())

In [ ]:
# ── Feature engineering: cyclic time, log rainfall, rolling gwl stats ─────────
# Cyclic encoding — month/day_of_year are circular, raw integers confuse the model
df['month_sin'] = np.sin(2*np.pi * df['month']       / 12.0)
df['month_cos'] = np.cos(2*np.pi * df['month']       / 12.0)
df['doy_sin']   = np.sin(2*np.pi * df['day_of_year'] / 365.25)
df['doy_cos']   = np.cos(2*np.pi * df['day_of_year'] / 365.25)

# Log-transform heavy-tailed rainfall (many zeros, occasional huge values)
for c in ['rainfall_mm','rainfall_7d','rainfall_30d']:
    df[c+'_log'] = np.log1p(df[c].clip(lower=0))

# Rolling GWL statistics per district — captures local volatility
for w in [30, 60, 90]:
    df[f'gwl_roll_mean_{w}'] = df.groupby('district')['gwl_mean'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=5).mean())
    df[f'gwl_roll_std_{w}']  = df.groupby('district')['gwl_mean'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=5).std())

# fill any rolling NaNs at series start with per-district median
roll_cols = [c for c in df.columns if c.startswith('gwl_roll_')]
df[roll_cols] = df.groupby('district')[roll_cols].transform(lambda s: s.fillna(s.median()))
df[roll_cols] = df[roll_cols].fillna(0)

print('Engineered features: 4 cyclic + 3 log_rainfall + 6 rolling_gwl =', 4+3+6, 'new cols')
print('Total cols:', df.shape[1], '  Missing:', df.isnull().sum().sum())

In [ ]:
# ── Feature definitions ───────────────────────────────────────────────────────
STATIC_CAT  = ['district_id', 'aquifer_category_id']   # +CGWB aquifer-stress category
STATIC_CONT = SOIL_COLS + [
    # CGWB extraction / recharge features — capture human water demand,
    # the missing driver for over-exploited districts (Jalgaon, Aurangabad, Nashik)
    'annual_recharge_bcm',
    'annual_extraction_bcm',
    'extractable_resource_bcm',
    'stage_of_extraction_pct',
]
PAST_CONT   = [
    'gwl_mean','gwl_min','gwl_max','station_count',
    'rainfall_mm_log','rainfall_7d_log','rainfall_30d_log',
    'et_mm','soil_moisture','net_water_balance',
    'temp_max','temp_min','temp_mean','humidity_mean','vpd_approx',
    'gwl_lag_7','gwl_lag_14','gwl_lag_30','gwl_7d_change','gwl_30d_change',
    'gwl_roll_mean_30','gwl_roll_std_30','gwl_roll_mean_60','gwl_roll_std_60',
    'gwl_roll_mean_90','gwl_roll_std_90',
    'ndvi_mean','ndwi_mean','ndmi_mean','vv_db','vh_db','vh_vv_ratio',
]
# Future features: known calendar values. Use cyclic + is_monsoon + season_id
FUTURE_CONT = ['month_sin','month_cos','doy_sin','doy_cos','is_monsoon','season_id']
TARGET      = 'gwl_mean'
N_DISTRICTS = df['district_id'].nunique()
N_CATEGORIES = int(df['aquifer_category_id'].max()) + 1   # 0..N-1 for embedding
print(f'Past={len(PAST_CONT)}, Future={len(FUTURE_CONT)}, Static={len(STATIC_CONT)}, Districts={N_DISTRICTS}, Categories={N_CATEGORIES}')

In [ ]:
# ── Normalise — PER-DISTRICT target normalization (key change) ───────────────
train_mask = df['date'] < TRAIN_END

# Feature scaler: global (features can share scale)
feature_cols = PAST_CONT + STATIC_CONT + FUTURE_CONT
# remove target from features — we'll scale it separately
feature_cols_no_target = [c for c in feature_cols if c != TARGET]

feature_scaler = StandardScaler()
feature_scaler.fit(df.loc[train_mask, feature_cols_no_target])
df_norm = df.copy()
df_norm[feature_cols_no_target] = feature_scaler.transform(df[feature_cols_no_target])

# Per-district target stats from TRAIN data only
tgt_stats = (df.loc[train_mask].groupby('district')[TARGET]
             .agg(['mean','std']).rename(columns={'mean':'mu','std':'sd'}))
tgt_stats['sd'] = tgt_stats['sd'].replace(0, 1).fillna(1)
tgt_stats.to_csv(OUT_DIR / 'per_district_target_stats.csv')

# Apply per-district z-score to BOTH target and gwl_* features that feed back as lags
gwl_like_cols = ['gwl_mean','gwl_min','gwl_max',
                 'gwl_lag_7','gwl_lag_14','gwl_lag_30',
                 'gwl_roll_mean_30','gwl_roll_mean_60','gwl_roll_mean_90']
# gwl_*_change and gwl_roll_std_* are differences/spreads — scale only by std (no shift)
gwl_diff_cols = ['gwl_7d_change','gwl_30d_change',
                 'gwl_roll_std_30','gwl_roll_std_60','gwl_roll_std_90']

# Restore original (unscaled) values for those cols, then apply per-district z
for c in gwl_like_cols + gwl_diff_cols:
    df_norm[c] = df[c]

dist_mu = df_norm['district'].map(tgt_stats['mu'])
dist_sd = df_norm['district'].map(tgt_stats['sd'])
for c in gwl_like_cols:
    df_norm[c] = (df_norm[c] - dist_mu) / dist_sd
for c in gwl_diff_cols:
    df_norm[c] = df_norm[c] / dist_sd

print(f'Train: {train_mask.sum():,} rows  |  Val: {(~train_mask).sum():,} rows')
print(f'Per-district target stats saved. Sample:')
print(tgt_stats.head().to_string())

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class GWLDataset(Dataset):
    def __init__(self, df, encoder_len, decoder_len,
                 past_cols, future_cols, static_cat_cols, static_cont_cols,
                 target_col, split='train', train_end=TRAIN_END):
        self.windows = []
        for dist, grp in df.groupby('district'):
            grp = grp.sort_values('date').reset_index(drop=True)
            n   = len(grp)
            for i in range(encoder_len, n - decoder_len + 1):
                anchor = grp.loc[i-1, 'date']
                if split == 'train' and anchor >= pd.Timestamp(train_end): continue
                if split == 'val'   and anchor <  pd.Timestamp(train_end): continue
                pc = grp.iloc[i-encoder_len:i][past_cols].values.astype(np.float32)
                fc = grp.iloc[i:i+decoder_len][future_cols].values.astype(np.float32)
                sc = grp.loc[0, static_cat_cols].values.astype(np.int64)
                sx = grp.loc[0, static_cont_cols].values.astype(np.float32)
                tg = grp.iloc[i:i+decoder_len][target_col].values.astype(np.float32)
                if np.isnan(pc).any() or np.isnan(fc).any() or np.isnan(tg).any(): continue
                self.windows.append({'past_cont':pc,'future_cont':fc,'static_cat':sc,
                                     'static_cont':sx,'target':tg,'district':dist,
                                     'date':grp.iloc[i+decoder_len-1]['date']})
    def __len__(self): return len(self.windows)
    def __getitem__(self, idx):
        w = self.windows[idx]
        return (torch.from_numpy(w['static_cat']), torch.from_numpy(w['static_cont']),
                torch.from_numpy(w['past_cont']),  torch.from_numpy(w['future_cont']),
                torch.from_numpy(w['target']))

print('Building datasets...')
t0 = time.time()
train_ds = GWLDataset(df_norm, ENCODER_LEN, DECODER_LEN, PAST_CONT, FUTURE_CONT, STATIC_CAT, STATIC_CONT, TARGET, 'train')
val_ds   = GWLDataset(df_norm, ENCODER_LEN, DECODER_LEN, PAST_CONT, FUTURE_CONT, STATIC_CAT, STATIC_CONT, TARGET, 'val')
print(f'Done in {time.time()-t0:.1f}s  |  Train: {len(train_ds):,}  |  Val: {len(val_ds):,}')

pin = DEVICE.type == 'cuda'
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=pin)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin)

In [ ]:
# ── Model + custom loss (q50-weighted quantile + huber blend) ─────────────────
# Two categorical embeddings now: district_id (N=28) + aquifer_category_id (N=3)
model = TemporalFusionTransformer(
    n_static_cat=len(STATIC_CAT), cat_vocab_sizes=[N_DISTRICTS, N_CATEGORIES], cat_embed_dim=CAT_EMBED_DIM,
    n_static_cont=len(STATIC_CONT), n_past=len(PAST_CONT), n_future=len(FUTURE_CONT),
    encoder_len=ENCODER_LEN, decoder_len=DECODER_LEN,
    d_model=D_MODEL, n_heads=N_HEADS, lstm_layers=LSTM_LAYERS,
    dropout=DROPOUT, quantiles=QUANTILES,
).to(DEVICE)

class WeightedQuantileLoss(nn.Module):
    """Pinball loss with extra weight on the median (q50). Median is what we
    report as the point forecast — biasing the loss towards it tightens MAE/RMSE
    without sacrificing useful coverage from q10/q90."""
    def __init__(self, quantiles, q50_boost=1.5):
        super().__init__()
        self.quantiles = quantiles
        w = [q50_boost if abs(q-0.5) < 1e-6 else 1.0 for q in quantiles]
        self.register_buffer('w', torch.tensor(w).float())
    def forward(self, preds, target):
        # preds: (B, T, Q), target: (B, T)
        target = target.unsqueeze(-1)
        diffs  = target - preds
        losses = []
        for i, q in enumerate(self.quantiles):
            d = diffs[..., i]
            losses.append(self.w[i] * torch.max(q*d, (q-1)*d).mean())
        return torch.stack(losses).sum()

criterion = WeightedQuantileLoss(QUANTILES, q50_boost=Q50_LOSS_BOOST).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# Cosine LR with linear warmup
steps_per_epoch = len(train_loader)
total_steps     = MAX_EPOCHS * steps_per_epoch
warmup_steps    = WARMUP_EPOCHS * steps_per_epoch
def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * min(p, 1.0)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters : {n_params:,}')
print(f'Device     : {DEVICE}')
print(f'Static cat : {STATIC_CAT}  vocabs=[{N_DISTRICTS}, {N_CATEGORIES}]')
print(f'Loss       : WeightedQuantileLoss(q50_boost={Q50_LOSS_BOOST})')
print(f'Scheduler  : Cosine w/ {WARMUP_EPOCHS}-epoch warmup')

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
def run_epoch(loader, model, criterion, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train(training)
    total, n = 0.0, 0
    with torch.set_grad_enabled(training):
        for batch in loader:
            sc, sx, px, fx, y = [b.to(DEVICE) for b in batch]
            preds, _, _ = model(sc, sx, px, fx)
            loss = criterion(preds, y)
            if training:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                if scheduler is not None: scheduler.step()
            total += loss.item(); n += 1
    return total / n

train_losses, val_losses = [], []
best_val, patience_ctr, best_state = float('inf'), 0, None

print(f'Training | {len(train_loader)} batches/epoch | patience={PATIENCE}')
print('-'*60)
for epoch in range(1, MAX_EPOCHS+1):
    t0  = time.time()
    tr  = run_epoch(train_loader, model, criterion, optimizer, scheduler)
    val = run_epoch(val_loader,   model, criterion)
    train_losses.append(tr); val_losses.append(val)
    print(f'Ep {epoch:03d} | train={tr:.4f} val={val:.4f} | lr={optimizer.param_groups[0]["lr"]:.1e} | {time.time()-t0:.1f}s')
    if val < best_val:
        best_val = val; patience_ctr = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'Early stop (best val={best_val:.4f})')
            break

model.load_state_dict(best_state)
torch.save(best_state, OUT_DIR / 'tft_checkpoint.pt')
print('Saved -> outputs/tft_checkpoint.pt')

In [ ]:
# ── Training curve ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(train_losses, label='Train'); ax.plot(val_losses, label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Quantile loss'); ax.set_title('TFT Training Curve')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR/'tft_training_curve.png', dpi=150); plt.show()

In [ ]:
# ── Evaluation (per-district un-normalization) ────────────────────────────────
def evaluate(loader, model, eval_steps, dataset):
    model.eval(); records = []
    with torch.no_grad():
        for bi, batch in enumerate(loader):
            sc, sx, px, fx, y = [b.to(DEVICE) for b in batch]
            preds, _, _ = model(sc, sx, px, fx)
            B = y.shape[0]
            for step in eval_steps:
                h = step + 1
                for i in range(B):
                    w = dataset.windows[bi*BATCH_SIZE+i]
                    mu = tgt_stats.loc[w['district'],'mu']
                    sd = tgt_stats.loc[w['district'],'sd']
                    act = float(y[i,step].cpu().numpy())            * sd + mu
                    q10 = float(preds[i,step,0].cpu().numpy())      * sd + mu
                    q50 = float(preds[i,step,1].cpu().numpy())      * sd + mu
                    q90 = float(preds[i,step,2].cpu().numpy())      * sd + mu
                    records.append({'district':w['district'],'date':str(w['date']),'horizon':f't+{h}',
                                    'actual':act,'q10':q10,'q50':q50,'q90':q90})
    return pd.DataFrame(records)

def metrics(df_pred, group=None):
    keys = ['horizon'] if group is None else [group,'horizon']
    rows = []
    for k, g in df_pred.groupby(keys):
        a, p = g['actual'].values, g['q50'].values
        covered = ((g['actual'] >= g['q10']) & (g['actual'] <= g['q90'])).mean()
        interval_width = (g['q90'] - g['q10']).mean()
        row = {'RMSE (m)':round(np.sqrt(np.mean((a-p)**2)),4),
               'MAE (m)':round(np.mean(np.abs(a-p)),4),
               'MAPE (%)':round(np.mean(np.abs((a-p)/(np.abs(a)+1e-8)))*100,2),
               'Coverage (%)':round(covered*100,1),
               'Interval (m)':round(interval_width,3),
               'n':len(g)}
        if group is None: row['Horizon'] = k
        else:             row[group], row['Horizon'] = k[0], k[1]
        rows.append(row)
    return pd.DataFrame(rows)

val_preds   = evaluate(val_loader, model, EVAL_STEPS, val_ds)
val_metrics = metrics(val_preds).set_index('Horizon').sort_index()
print('\n=== TFT Aggregate Results ===')
print(val_metrics.to_string())
print('\nNote: Coverage (%) = % of actuals inside 80% interval [q10, q90]. Target: ~80%.')
print('      Interval (m) = mean q90−q10 width. Wider = more uncertain.')
val_preds.to_csv(OUT_DIR/'tft_predictions.csv', index=False)
val_metrics.to_csv(OUT_DIR/'tft_metrics.csv')

In [ ]:
# ── PER-DISTRICT METRICS ──────────────────────────────────────────────────────
per_dist = metrics(val_preds, group='district')
per_dist = per_dist[['district','Horizon','RMSE (m)','MAE (m)','MAPE (%)','Coverage (%)','Interval (m)','n']]
per_dist = per_dist.sort_values(['Horizon','MAPE (%)']).reset_index(drop=True)

print('\n=== PER-DISTRICT RESULTS ===\n')
for h in ['t+7','t+14','t+30']:
    sub = per_dist[per_dist['Horizon']==h].drop(columns='Horizon').reset_index(drop=True)
    print(f'--- {h} ---')
    print(sub.to_string(index=False))
    print(f'   MEAN  : RMSE={sub["RMSE (m)"].mean():.3f}  MAE={sub["MAE (m)"].mean():.3f}  MAPE={sub["MAPE (%)"].mean():.2f}%  Coverage={sub["Coverage (%)"].mean():.1f}%')
    print(f'   MEDIAN: RMSE={sub["RMSE (m)"].median():.3f}  MAE={sub["MAE (m)"].median():.3f}  MAPE={sub["MAPE (%)"].median():.2f}%  Coverage={sub["Coverage (%)"].median():.1f}%')
    print()

per_dist.to_csv(OUT_DIR/'tft_metrics_per_district.csv', index=False)
print('Saved -> outputs/tft_metrics_per_district.csv')

In [ ]:
# ── COVERAGE RATE ANALYSIS ────────────────────────────────────────────────────
# Coverage = % of actual values that fall inside the 80% prediction interval [q10, q90]
# Well-calibrated model: ~80%. Undercoverage (<70%) = intervals too tight.
# Overcoverage (>90%) = intervals too wide / model is being overly conservative.

val_preds['covered'] = (val_preds['actual'] >= val_preds['q10']) & (val_preds['actual'] <= val_preds['q90'])
val_preds['interval_width'] = val_preds['q90'] - val_preds['q10']

print('=== COVERAGE RATE ANALYSIS ===')
print('Target: ~80% of actuals should fall inside the [q10, q90] interval\n')

# Aggregate by horizon
agg_cov = val_preds.groupby('horizon').agg(
    coverage=('covered', lambda x: round(x.mean()*100,1)),
    mean_interval_width=('interval_width', lambda x: round(x.mean(),3)),
    n=('covered','count')
).reset_index()
print('Aggregate coverage by horizon:')
print(agg_cov.to_string(index=False))

# Per-district coverage pivot (districts as rows, horizons as columns)
print('\nPer-district coverage (%) — target ~80 in each cell:')
cov_pivot = (val_preds.groupby(['district','horizon'])['covered']
             .mean().mul(100).round(1).unstack('horizon'))
cov_pivot.columns.name = None
print(cov_pivot.to_string())

# Flag poorly-calibrated districts
for h in cov_pivot.columns:
    undercovered = cov_pivot[cov_pivot[h] < 65].index.tolist()
    overcovered  = cov_pivot[cov_pivot[h] > 95].index.tolist()
    if undercovered: print(f'\n  {h} UNDERCOVERED (<65%): {undercovered}')
    if overcovered:  print(f'\n  {h} OVERCOVERED (>95%): {overcovered}')

# Save coverage pivot to CSV
cov_pivot.to_csv(OUT_DIR/'tft_coverage_by_district.csv')
print('\nSaved -> outputs/tft_coverage_by_district.csv')

# ── Coverage heatmap ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, max(6, len(cov_pivot)*0.35)))
im = ax.imshow(cov_pivot.values, aspect='auto', vmin=50, vmax=100,
               cmap='RdYlGn', interpolation='nearest')
ax.set_xticks(range(len(cov_pivot.columns)))
ax.set_xticklabels(cov_pivot.columns)
ax.set_yticks(range(len(cov_pivot.index)))
ax.set_yticklabels(cov_pivot.index, fontsize=8)
ax.set_title('80% Interval Coverage Rate (%) per District\n(green ≈ 80% = well-calibrated, red = poorly calibrated)')
plt.colorbar(im, ax=ax, label='Coverage (%)')
# Annotate each cell with the value
for i in range(len(cov_pivot.index)):
    for j in range(len(cov_pivot.columns)):
        v = cov_pivot.values[i, j]
        ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=7,
                color='black' if 65 < v < 95 else 'white')
plt.tight_layout()
plt.savefig(OUT_DIR/'tft_coverage_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ── CONFORMAL CALIBRATION (fixes undercoverage without retraining) ────────────
# For each sample, compute how far the actual falls outside [q10, q90].
# The 80th percentile of those scores becomes a margin added symmetrically
# to every interval, guaranteeing ~80% empirical coverage.

TARGET_COVERAGE = 0.80

cal_preds = val_preds.copy()

# Nonconformity score: positive if actual is outside the interval, else negative
cal_preds['nc_score'] = np.maximum(
    cal_preds['q10'] - cal_preds['actual'],   # actual below q10
    cal_preds['actual'] - cal_preds['q90']    # actual above q90
)

# Per-horizon margin = 80th percentile of the nonconformity scores
print('=== CONFORMAL CALIBRATION MARGINS ===')
margins = {}
for h, g in cal_preds.groupby('horizon'):
    n = len(g)
    # Conformal quantile level (finite-sample correction)
    level = np.ceil((n + 1) * TARGET_COVERAGE) / n
    level = min(level, 1.0)
    margin = float(np.quantile(g['nc_score'], level))
    margins[h] = margin
    print(f'  {h}: margin = {margin:+.4f} m  (applied to every q10/q90 for this horizon)')

# Apply margins: widen q10 down and q90 up by the margin
cal_preds['q10_cal'] = cal_preds.apply(lambda r: r['q10'] - margins[r['horizon']], axis=1)
cal_preds['q90_cal'] = cal_preds.apply(lambda r: r['q90'] + margins[r['horizon']], axis=1)

# Recheck coverage after calibration
cal_preds['covered_cal'] = (cal_preds['actual'] >= cal_preds['q10_cal']) & \
                            (cal_preds['actual'] <= cal_preds['q90_cal'])
cal_preds['width_cal'] = cal_preds['q90_cal'] - cal_preds['q10_cal']

print('\nCoverage after calibration:')
check = cal_preds.groupby('horizon').agg(
    before=('covered', lambda x: round(x.mean()*100,1)),
    after=('covered_cal', lambda x: round(x.mean()*100,1)),
    mean_width_before=('interval_width', lambda x: round(x.mean(),3)),
    mean_width_after=('width_cal', lambda x: round(x.mean(),3)),
).reset_index()
print(check.to_string(index=False))

# Save calibrated predictions
cal_preds.to_csv(OUT_DIR/'tft_predictions_calibrated.csv', index=False)
print('\nSaved -> outputs/tft_predictions_calibrated.csv')
print('Columns: q10_cal and q90_cal are the calibrated bounds to use on the dashboard.')

In [ ]:
# ── Best / worst districts bar plot ──────────────────────────────────────────
for h in ['t+7','t+14','t+30']:
    sub = per_dist[per_dist['Horizon']==h].sort_values('MAPE (%)')
    fig, ax = plt.subplots(figsize=(12,6))
    colors = ['seagreen' if v < sub['MAPE (%)'].median() else 'indianred' for v in sub['MAPE (%)']]
    ax.barh(sub['district'], sub['MAPE (%)'], color=colors)
    ax.set_xlabel('MAPE (%)'); ax.set_title(f'Per-district MAPE @ {h}')
    ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR/f'per_district_mape_{h.replace("+","")}.png', dpi=150)
    plt.show()

In [ ]:
# ── Quantile plots (top 4 districts by sample count) ─────────────────────────
top_dists = val_preds['district'].value_counts().head(4).index.tolist()
for dist in top_dists:
    sub = val_preds[(val_preds['district']==dist) & (val_preds['horizon']=='t+30')].sort_values('date').tail(90)
    x = range(len(sub))
    fig, ax = plt.subplots(figsize=(14,4))
    ax.fill_between(x, sub['q10'], sub['q90'], alpha=0.2, color='steelblue', label='80% interval')
    ax.plot(x, sub['q50'],    color='steelblue', lw=1.5, label='q50 forecast')
    ax.plot(x, sub['actual'], color='black',    lw=1.5, ls='--', label='Actual GWL')
    ax.set_title(f'{dist} — t+30'); ax.set_ylabel('GWL (m)'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT_DIR/f'quantile_{dist.lower().replace(" ","_")}.png', dpi=150); plt.show()

In [ ]:
# ── VSN feature importance ────────────────────────────────────────────────────
model.eval()
pw, fw, sw = [], [], []
with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i >= 30: break
        sc, sx, px, fx, _ = [b.to(DEVICE) for b in batch]
        _, vsn_w, _ = model(sc, sx, px, fx)
        pw.append(vsn_w['past'].cpu().mean(dim=(0,1)))
        fw.append(vsn_w['future'].cpu().mean(dim=(0,1)))
        sw.append(vsn_w['static'].cpu().mean(dim=0))
past_imp = torch.stack(pw).mean(0).numpy()
fut_imp  = torch.stack(fw).mean(0).numpy()
stat_imp = torch.stack(sw).mean(0).numpy()

# Static features: cat embeddings first (one per STATIC_CAT), then continuous
static_labels = [f'{c}_emb' for c in STATIC_CAT] + STATIC_CONT

fig, axes = plt.subplots(1,3,figsize=(20,8))
for ax, imp, names, color, title in [
    (axes[0], past_imp, PAST_CONT, 'steelblue', 'Past Features (VSN)'),
    (axes[1], fut_imp,  FUTURE_CONT,'coral',    'Future Features (VSN)'),
    (axes[2], stat_imp, static_labels, 'seagreen', 'Static Features (VSN)'),
]:
    idx = np.argsort(imp)[::-1]
    ax.barh([names[i] for i in idx], imp[idx], color=color)
    ax.set_title(title); ax.set_xlabel('Mean VSN weight'); ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUT_DIR/'tft_vsn_importance.png', dpi=150); plt.show()

In [ ]:
# ── Download all outputs ──────────────────────────────────────────────────────
import shutil
shutil.make_archive('tft_outputs', 'zip', 'outputs')
files.download('tft_outputs.zip')
files.download('outputs/tft_checkpoint.pt')
print('Downloaded: tft_outputs.zip  +  tft_checkpoint.pt')